# LabsX – Basic: Introduction to Web Scraping
ดึงชื่อหนังสือและรายชื่อบทจาก https://automatetheboringstuff.com/3e/ ด้วย `requests` + `BeautifulSoup4`

**รัน cell ตามลำดับจากบนลงล่าง**

In [1]:
!pip install -q requests beautifulsoup4
!mkdir -p src data

## 1. `src/scraper.py` — ส่วนดึงข้อมูลจากเว็บ

In [2]:
%%writefile src/scraper.py
# web-scraping-intro/src/scraper.py
import re
import time
from urllib import robotparser
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
)

# ชื่อบทในหน้านี้ขึ้นต้นด้วยคำเหล่านี้ เช่น "Chapter 12 – Web Scraping"
CHAPTER_PATTERN = re.compile(r"^(Introduction|Chapter\s+\d+|Appendix\s+[A-Z])", re.IGNORECASE)


class SimpleWebScraper:
    """
    A class to encapsulate basic web scraping functionality.
    """

    def __init__(self, target_url, delay=1.0):
        self.target_url = target_url
        self.delay = delay  # หน่วงเวลาก่อนส่ง request เพื่อไม่ให้รบกวน server (rate limiting)
        self.headers = {"User-Agent": USER_AGENT}

    def is_allowed_by_robots(self):
        """
        Check robots.txt to see whether we are allowed to scrape the target URL.
        If robots.txt cannot be read, assume it is allowed.
        """
        parts = urlparse(self.target_url)
        robots_url = f"{parts.scheme}://{parts.netloc}/robots.txt"
        rp = robotparser.RobotFileParser()
        try:
            response = requests.get(robots_url, headers=self.headers, timeout=10)
            if response.status_code != 200:
                return True
            rp.parse(response.text.splitlines())
            return rp.can_fetch("*", self.target_url)
        except requests.exceptions.RequestException:
            return True

    def _get_html_content(self):
        """
        Downloads the HTML content from the target URL.
        Handles basic error checking.
        """
        print(f"กำลังดาวน์โหลดเนื้อหาจาก: {self.target_url}")
        time.sleep(self.delay)
        try:
            response = requests.get(self.target_url, headers=self.headers, timeout=10)
            response.raise_for_status()  # Raise an HTTPError for 4xx/5xx responses
            print("ดาวน์โหลดเนื้อหาสำเร็จ")
            return response.text
        except requests.exceptions.HTTPError as e:
            print(f"เกิดข้อผิดพลาด HTTP: {e} - Status Code: {e.response.status_code}")
        except requests.exceptions.ConnectionError:
            print(f"เชื่อมต่อไม่ได้: {self.target_url}")
        except requests.exceptions.Timeout:
            print("หมดเวลาในการเชื่อมต่อ (timeout)")
        except requests.exceptions.RequestException as e:
            print(f"เกิดข้อผิดพลาดที่ไม่คาดคิด: {e}")
        return None

    @staticmethod
    def parse_book_title(soup):
        """
        Find the book title from <article> -> <header> -> <h1>.
        Fall back to the first <h1>, then the <title> tag.
        Always check for None before calling get_text().
        """
        article_tag = soup.find("article")
        if article_tag:
            header_tag = article_tag.find("header")
            if header_tag:
                h1_tag = header_tag.find("h1")
                if h1_tag and h1_tag.get_text(strip=True):
                    return h1_tag.get_text(strip=True)
        h1 = soup.find("h1")
        if h1 and h1.get_text(strip=True):
            return h1.get_text(" ", strip=True)
        if soup.title and soup.title.string:
            return soup.title.string.strip()
        return "Book Title Not Found"

    def parse_chapters(self, soup):
        """
        Find every link whose text looks like a chapter title
        (Introduction, Chapter N, Appendix X) and return title + full URL.
        """
        chapters = []
        seen = set()
        for link in soup.find_all("a", href=True):
            title = link.get_text(" ", strip=True)
            if not CHAPTER_PATTERN.match(title) or title in seen:
                continue
            seen.add(title)
            chapters.append({
                "title": title,
                "url": urljoin(self.target_url, link["href"]),
            })
        return chapters

    def scrape_main_titles(self):
        """
        Scrapes the main book title and chapter titles from the Automate the Boring Stuff homepage.
        """
        if not self.is_allowed_by_robots():
            print("robots.txt ไม่อนุญาตให้ scrape หน้านี้ หยุดการทำงาน")
            return None

        html_content = self._get_html_content()
        if not html_content:
            return None

        soup = BeautifulSoup(html_content, "html.parser")
        return {
            "source_url": self.target_url,
            "book_title": self.parse_book_title(soup),
            "chapters": self.parse_chapters(soup),
        }


Writing src/scraper.py


## 2. `src/data_processor.py` — บันทึกผลเป็น JSON

In [3]:
%%writefile src/data_processor.py
# web-scraping-intro/src/data_processor.py
import json
import os


def save_to_json(data, filepath):
    """Save scraped data to a JSON file (creates the folder if needed)."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


Writing src/data_processor.py


In [4]:
%%writefile src/__init__.py
# web-scraping-intro/src/__init__.py
# This file indicates that the 'src' directory is a Python package.


Writing src/__init__.py


## 3. `main.py` — จุดเริ่มต้นของโปรแกรม

In [5]:
%%writefile main.py
# web-scraping-intro/main.py
import os
import sys

# Add the 'src' directory to the Python path
sys.path.append(os.path.join(os.path.dirname(os.path.abspath(__file__)), "src"))

from scraper import SimpleWebScraper
from data_processor import save_to_json

OUTPUT_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)), "data", "scraped_output.json")


def main():
    """
    Main entry point for the simple web scraping application.
    """
    target_url = "https://automatetheboringstuff.com/3e/"
    scraper = SimpleWebScraper(target_url)

    scraped_data = scraper.scrape_main_titles()

    if not scraped_data:
        print("ไม่สามารถ scrape ข้อมูลได้")
        return

    print("\n--- ข้อมูลที่ Scrape ได้ ---")
    print(f"ชื่อหนังสือ: {scraped_data['book_title']}")
    print("\nชื่อบทต่างๆ:")
    if scraped_data["chapters"]:
        for i, chapter in enumerate(scraped_data["chapters"], start=1):
            print(f"{i}. {chapter['title']}")
    else:
        print("ไม่พบชื่อบท")
    print("--------------------")

    save_to_json(scraped_data, OUTPUT_FILE)


if __name__ == "__main__":
    main()


Writing main.py


## 4. รันโปรแกรม

In [6]:
!python main.py

กำลังดาวน์โหลดเนื้อหาจาก: https://automatetheboringstuff.com/3e/
ดาวน์โหลดเนื้อหาสำเร็จ

--- ข้อมูลที่ Scrape ได้ ---
ชื่อหนังสือ: 3rd Edition

ชื่อบทต่างๆ:
1. Introduction
2. Chapter 1 - Python Basics
3. Chapter 2 - if-else and Flow Control
4. Chapter 3 - Loops
5. Chapter 4 - Functions
6. Chapter 5 - Debugging
7. Chapter 6 - Lists
8. Chapter 7 - Dictionaries and Structuring Data
9. Chapter 8 - Strings and Text Editing
10. Chapter 9 - Text Pattern Matching with Regular Expressions
11. Chapter 10 - Reading and Writing Files
12. Chapter 11 - Organizing Files
13. Chapter 12 - Designing and Deploying Command Line Programs
14. Chapter 13 - Web Scraping
15. Chapter 14 - Excel Spreadsheets
16. Chapter 15 - Google Sheets
17. Chapter 16 - SQLite Databases
18. Chapter 17 - PDF and Word Documents
19. Chapter 18 - CSV, JSON, and XML Files
20. Chapter 19 - Keeping Time, Scheduling Tasks, and Launching Programs
21. Chapter 20 - Sending Email, Texts, and Push Notifications
22. Chapter 21 - Making Gra

## 5. ดูไฟล์ JSON ที่บันทึกไว้

In [7]:
!head -n 20 data/scraped_output.json

{
  "source_url": "https://automatetheboringstuff.com/3e/",
  "book_title": "3rd Edition",
  "chapters": [
    {
      "title": "Introduction",
      "url": "https://automatetheboringstuff.com/3e/chapter0.html"
    },
    {
      "title": "Chapter 1 - Python Basics",
      "url": "https://automatetheboringstuff.com/3e/chapter1.html"
    },
    {
      "title": "Chapter 2 - if-else and Flow Control",
      "url": "https://automatetheboringstuff.com/3e/chapter2.html"
    },
    {
      "title": "Chapter 3 - Loops",
      "url": "https://automatetheboringstuff.com/3e/chapter3.html"
    },
